# Stage 5 — Blocked Spatial Cross-Validation

## Why spatial CV?

Random k-fold inflates AUC because training and test points sit next to each other — the model memorises **spatial autocorrelation** rather than genuine habitat preference.  Blocked spatial CV holds out entire geographic blocks, so the test set is spatially independent of training, giving an honest performance estimate.

## Method

The San Diego study area is divided into a **5×5 regular grid** (25 blocks).  Each non-empty block is held out once while the remaining blocks form training.  Per fold:
1. Scaler is refitted on training blocks ONLY (no leakage)
2. A throw-away model is trained with early stopping
3. Per-species AUC is computed on the held-out block

**Output:** `models/species_auc_spatial_cv.json` and `models/species_labels_filtered.json`

Species with mean spatial CV AUC < `MIN_SPECIES_AUC` (0.65) are flagged — the inference layer (Stage 6) can optionally filter them from API responses.

In [ ]:
import sys
from pathlib import Path
_here = Path.cwd()
_root = _here.parent if _here.name == 'notebooks' else _here
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)-8s %(name)s  %(message)s',
    force=True,
)

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import torch

from pipeline.features import build_features, build_label_matrix_fast, get_species_list
from pipeline.model import SDMModel, compute_species_auc, load_species_labels
from pipeline.evaluation import (
    assign_blocks, run_spatial_cv, aggregate_cv_auc,
    save_cv_auc, load_cv_auc, save_filtered_species,
)
from config import (
    SAMPLED_PATH, MODELS_DIR, SCALER_PATH, MODEL_PATH, SPECIES_LABELS_PATH,
    CV_GRID_SIZE, MIN_SPECIES_AUC, SD_LAT_MIN, SD_LAT_MAX, SD_LON_MIN, SD_LON_MAX,
)

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 110})

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print('Setup complete')

## 1. Load data and inspect block distribution

In [ ]:
df = pd.read_parquet(SAMPLED_PATH)
X = build_features(df)
species_list = get_species_list(df)
Y = build_label_matrix_fast(df, species_list)
S = len(species_list)

print(f'Rows: {len(df):,}  |  Species: {S}  |  Positives: {int(Y.sum()):,}')

# Use data-driven extent — the full SD bounding box is much larger than the
# campus study area, which would put every row in a single block.
buf = 1e-4
DATA_LAT_MIN = df['lat'].min() - buf
DATA_LAT_MAX = df['lat'].max() + buf
DATA_LON_MIN = df['lon'].min() - buf
DATA_LON_MAX = df['lon'].max() + buf
print(f'Data extent: lat [{DATA_LAT_MIN:.4f}, {DATA_LAT_MAX:.4f}]  lon [{DATA_LON_MIN:.4f}, {DATA_LON_MAX:.4f}]')

blocks = assign_blocks(
    df['lat'].values, df['lon'].values,
    n_lat=CV_GRID_SIZE, n_lon=CV_GRID_SIZE,
    lat_min=DATA_LAT_MIN, lat_max=DATA_LAT_MAX,
    lon_min=DATA_LON_MIN, lon_max=DATA_LON_MAX,
)
unique_blocks, block_counts = np.unique(blocks, return_counts=True)
print(f'Non-empty blocks: {len(unique_blocks)} / {CV_GRID_SIZE**2}')
print(f'Rows per block: min={block_counts.min()}, median={np.median(block_counts):.0f}, max={block_counts.max()}')

In [ ]:
# Visualise block grid and sample density
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Block membership map
ax = axes[0]
ax.scatter(df['lon'], df['lat'], c=blocks, cmap='tab20',
           s=0.5, alpha=0.4, rasterized=True)

# Draw grid lines using data extent
lat_edges = np.linspace(DATA_LAT_MIN, DATA_LAT_MAX, CV_GRID_SIZE + 1)
lon_edges = np.linspace(DATA_LON_MIN, DATA_LON_MAX, CV_GRID_SIZE + 1)
for lat_e in lat_edges:
    ax.axhline(lat_e, color='black', lw=0.5, alpha=0.5)
for lon_e in lon_edges:
    ax.axvline(lon_e, color='black', lw=0.5, alpha=0.5)

ax.set_title(f'Spatial block assignment ({CV_GRID_SIZE}×{CV_GRID_SIZE} grid, data extent)')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')

# Block sample counts heatmap
ax2 = axes[1]
grid_counts = np.zeros((CV_GRID_SIZE, CV_GRID_SIZE))
for bid, cnt in zip(unique_blocks, block_counts):
    row = int(bid // CV_GRID_SIZE)
    col = int(bid % CV_GRID_SIZE)
    grid_counts[row, col] = cnt

im = ax2.imshow(grid_counts, cmap='YlOrRd', origin='lower', aspect='auto')
plt.colorbar(im, ax=ax2, label='Row count')
for r in range(CV_GRID_SIZE):
    for c in range(CV_GRID_SIZE):
        if grid_counts[r, c] > 0:
            ax2.text(c, r, f'{int(grid_counts[r,c]):,}', ha='center', va='center',
                     fontsize=7, color='black' if grid_counts[r,c] < grid_counts.max()*0.7 else 'white')

ax2.set_title('Rows per geographic block')
ax2.set_xlabel(f'Lon block (W→E)  [{DATA_LON_MIN:.4f}° to {DATA_LON_MAX:.4f}°]')
ax2.set_ylabel(f'Lat block (S→N)  [{DATA_LAT_MIN:.4f}° to {DATA_LAT_MAX:.4f}°]')

plt.tight_layout()
plt.show()

print('Blocks with < 50 rows (will be skipped as test folds):')
for bid, cnt in zip(unique_blocks, block_counts):
    if cnt < 50:
        print(f'  block={bid}  count={cnt}')

## 2. Run blocked spatial cross-validation

⏱ **Expected time: ~10–30 min on GPU depending on how many non-empty blocks qualify.**

Each fold trains a full model with early stopping (max 100 epochs, patience 8).  
The models are throw-away — only AUC scores are kept.

In [ ]:
CV_AUC_PATH = MODELS_DIR / 'species_auc_spatial_cv.json'

if CV_AUC_PATH.exists():
    print(f'Loading cached spatial CV results from {CV_AUC_PATH}')
    fold_aucs = load_cv_auc(CV_AUC_PATH)
    print('Done (cached).')
else:
    print('Running spatial CV (this may take a while)…')
    fold_aucs = run_spatial_cv(
        df, X, Y, species_list,
        n_grid=CV_GRID_SIZE,
        min_block_samples=50,
        min_positives=3,
        cv_max_epochs=100,
        cv_patience=8,
        device=DEVICE,
    )
    save_cv_auc(fold_aucs, CV_AUC_PATH)
    print('Spatial CV complete and saved.')

## 3. Aggregate and inspect per-species results

In [ ]:
summary = aggregate_cv_auc(fold_aucs)

valid = summary.dropna(subset=['mean_auc'])
print(f'Species evaluated (≥1 valid fold): {len(valid)}')
print(f'Mean spatial CV AUC : {valid["mean_auc"].mean():.3f}')
print(f'Median spatial CV AUC: {valid["mean_auc"].median():.3f}')
print(f'Species ≥ {MIN_SPECIES_AUC}: {summary["above_threshold"].sum()}')
print(f'Species < {MIN_SPECIES_AUC}: {(~summary["above_threshold"]).sum()}')
print()
print('Top 10:')
display(summary.head(10)[['mean_auc','std_auc','n_folds','above_threshold']])
print('\nBottom 10:')
display(summary.tail(10)[['mean_auc','std_auc','n_folds','above_threshold']])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

auc_vals = valid['mean_auc'].values

# Histogram
axes[0].hist(auc_vals, bins=20, color='steelblue', edgecolor='none', alpha=0.85)
axes[0].axvline(MIN_SPECIES_AUC, color='red', ls='--',
                label=f'Threshold ({MIN_SPECIES_AUC})')
axes[0].axvline(np.median(auc_vals), color='green', ls='--',
                label=f'Median ({np.median(auc_vals):.3f})')
axes[0].set_title('Spatial CV AUC distribution')
axes[0].set_xlabel('Mean spatial CV AUC'); axes[0].set_ylabel('Species')
axes[0].legend()

# Error bar plot (sorted by mean AUC)
sorted_summary = valid.sort_values('mean_auc')
y_pos = np.arange(len(sorted_summary))
colours = ['tomato' if not v else 'steelblue' for v in sorted_summary['above_threshold']]
axes[1].barh(y_pos, sorted_summary['mean_auc'], xerr=sorted_summary['std_auc'],
             color=colours, height=0.7, error_kw={'elinewidth': 0.5, 'alpha': 0.5})
axes[1].axvline(MIN_SPECIES_AUC, color='red', ls='--', lw=1)
short_names = [sp.split()[-1][:18] for sp in sorted_summary.index]
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(short_names, fontsize=4)
axes[1].set_xlabel('Mean Spatial CV AUC ± std')
axes[1].set_title('Per-species spatial CV AUC (red = below threshold)')

plt.suptitle('Blocked Spatial Cross-Validation Results', fontsize=12)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'spatial_cv_auc.png', bbox_inches='tight', dpi=120)
plt.show()

## 4. Spatial vs random AUC comparison — leakage check

If spatial CV AUC is substantially lower than random-split AUC (Stage 4), the model was exploiting **spatial autocorrelation** and the random AUC was misleadingly optimistic.  A large gap (>0.10) means the habitat signal is weak relative to the spatial clustering.

In [ ]:
RANDOM_AUC_PATH = MODELS_DIR / 'species_auc_random_split.json'

if RANDOM_AUC_PATH.exists():
    with open(RANDOM_AUC_PATH) as f:
        random_auc_raw = json.load(f)
    random_auc = {sp: (float('nan') if v is None else v) for sp, v in random_auc_raw.items()}
    
    # Merge
    compare_rows = []
    for sp in species_list:
        r = random_auc.get(sp, float('nan'))
        s = summary.loc[sp, 'mean_auc'] if sp in summary.index else float('nan')
        if not np.isnan(r) and not np.isnan(s):
            compare_rows.append({'species': sp, 'random_auc': r, 'spatial_cv_auc': s,
                                  'gap': r - s})
    
    compare_df = pd.DataFrame(compare_rows).sort_values('gap', ascending=False)
    
    print(f'Species with both AUC scores: {len(compare_df)}')
    print(f'Mean gap (random - spatial): {compare_df["gap"].mean():.3f}')
    print(f'Max gap                    : {compare_df["gap"].max():.3f}  ({compare_df.iloc[0]["species"]})')
    print(f'Negative gap (spatial > random): {(compare_df["gap"] < 0).sum()} species')
    print()
    
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(
        compare_df['random_auc'],
        compare_df['spatial_cv_auc'],
        c=compare_df['gap'], cmap='RdYlGn_r', s=25, alpha=0.7, vmin=-0.1, vmax=0.3
    )
    ax.plot([0.5, 1], [0.5, 1], 'k--', lw=1, label='Equal performance')
    ax.axhline(MIN_SPECIES_AUC, color='red', ls=':', lw=1, alpha=0.5)
    ax.axvline(MIN_SPECIES_AUC, color='red', ls=':', lw=1, alpha=0.5)
    ax.set_xlabel('Random-split AUC (Stage 4)')
    ax.set_ylabel('Spatial CV AUC (Stage 5)')
    ax.set_title('Random vs Spatial AUC\n(points below diagonal = random AUC was optimistic)')
    ax.legend()
    plt.colorbar(ax.collections[0], ax=ax, label='Gap (random − spatial)')
    plt.tight_layout()
    plt.savefig(MODELS_DIR / 'random_vs_spatial_auc.png', bbox_inches='tight', dpi=120)
    plt.show()
    
    gap_threshold = 0.10
    large_gap = compare_df[compare_df['gap'] > gap_threshold]
    if len(large_gap):
        print(f'⚠ {len(large_gap)} species have gap > {gap_threshold} (possible spatial autocorrelation inflation):')
        display(large_gap.head(10).to_string(index=False))
    else:
        print(f'✓ No species have gap > {gap_threshold} — spatial autocorrelation is not severely inflating AUC.')
else:
    print('Random AUC file not found — run 03_model.ipynb first.')

## 5. Species inclusion decision

In [ ]:
species_above = summary.index[summary['above_threshold']].tolist()
species_below = summary.index[~summary['above_threshold']].tolist()

# Species with no valid folds (NaN mean_auc) — treated as unknown, kept in model
# but flagged so the API can communicate uncertainty
species_no_eval = summary.index[summary['mean_auc'].isna()].tolist()

print(f'Species ≥ {MIN_SPECIES_AUC} (included in filtered list): {len(species_above)}')
print(f'Species < {MIN_SPECIES_AUC} (below threshold)          : {len(species_below)}')
print(f'Species not evaluated (no valid folds)                 : {len(species_no_eval)}')
print()
print('Note: The final model (Stage 4) still predicts all species.')
print('The filtered list tells Stage 6 which predictions are reliable.')

if species_below:
    print(f'\nBelow-threshold species ({len(species_below)}):')
    for sp in sorted(species_below):
        row = summary.loc[sp]
        auc_str = f'{row["mean_auc"]:.3f}' if not np.isnan(row['mean_auc']) else 'NaN'
        print(f'  {auc_str}  {sp}')

In [ ]:
# Save filtered species list
FILTERED_LABELS_PATH = MODELS_DIR / 'species_labels_filtered.json'
save_filtered_species(species_above, FILTERED_LABELS_PATH)

# Save full summary CSV for inspection
summary_path = MODELS_DIR / 'spatial_cv_summary.csv'
summary.to_csv(summary_path)
print(f'Saved summary CSV → {summary_path}')

print(f'\nFiltered species list: {len(species_above)} species')
print(f'Full species list    : {len(species_list)} species')

## 6. Summary report

In [ ]:
valid_summary = summary.dropna(subset=['mean_auc'])

sep = '=' * 64
lines = [
    '',
    sep,
    '  STAGE 5 — SPATIAL CV SUMMARY',
    sep,
    f'  Grid                     : {CV_GRID_SIZE}×{CV_GRID_SIZE}',
    f'  Non-empty blocks         : {len(unique_blocks)}',
    f'  Folds run (≥50 test rows): (see log above)',
    '',
    f'  Species evaluated        : {len(valid_summary)}',
    f'  Mean spatial CV AUC      : {valid_summary["mean_auc"].mean():.3f}',
    f'  Median spatial CV AUC    : {valid_summary["mean_auc"].median():.3f}',
    f'  ≥ {MIN_SPECIES_AUC} (included)         : {len(species_above)}',
    f'  < {MIN_SPECIES_AUC} (flagged)          : {len(species_below)}',
    '',
    '  Files saved:',
    '    species_auc_spatial_cv.json',
    '    species_labels_filtered.json',
    '    spatial_cv_summary.csv',
    '    spatial_cv_auc.png',
    sep,
    '',
    '  → Proceed to 05_inference.ipynb (Stage 6)',
    sep,
]
print('\n'.join(lines))